# Load Silver Layer

## crm_cust_info

### Explore the data

In [0]:
-- Visualize bronze table
SELECT
  cst_id,
  cst_key,
  cst_firstname,
  cst_lastname,
  cst_marital_status,
  cst_gndr,
  cst_create_date
FROM `workspace`.`bronze`.`crm_cust_info`
LIMIT 20
;

-- We have to check for
-- 1 Unique primary key
-- 2 Trailing and leading spaces
-- 3 Consistency of marital status and gender



### 1 Primary key check

In [0]:
-- Visualize primary key
SELECT
cst_id,
COUNT(*)
FROM `workspace`.`bronze`.`crm_cust_info`
GROUP BY cst_id
HAVING COUNT(*) > 1 OR cst_id IS NULL
;

In [0]:
-- Check for unique primary key
SELECT
  cst_id,
  COUNT(*)
FROM `workspace`.`bronze`.`crm_cust_info`
GROUP BY cst_id
HAVING COUNT(*) > 1 OR cst_id IS NULL
;


-- Delete Nulls and duplicates (choose most recent one)
-- Final query
SELECT
*
FROM (
SELECT
*,
row_number() OVER(PARTITION BY cst_id ORDER BY cst_create_date DESC) AS flag_last
FROM `workspace`.`bronze`.`crm_cust_info`
)t
WHERE flag_last = 1 AND cst_id IS NOT NULL
;

-- Check results
SELECT
  cst_id,
  COUNT(*)
FROM
  (
    SELECT
    *
    FROM (
    SELECT
    *,
    row_number() OVER(PARTITION BY cst_id ORDER BY cst_create_date DESC) AS flag_last
    FROM `workspace`.`bronze`.`crm_cust_info`
    )t
    WHERE flag_last = 1 AND cst_id IS NOT NULL
  )
GROUP BY cst_id
HAVING COUNT(*) > 1 OR cst_id IS NULL
;


### 2 Unwanted spaces in `cst_firstname` and `cst_lastname`

In [0]:
-- Check trailing and leading spaces
SELECT
  cst_firstname,
  cst_lastname
FROM `workspace`.`bronze`.`crm_cust_info`
WHERE
  cst_firstname != TRIM(cst_firstname) OR
  cst_lastname != TRIM(cst_lastname)
;

-- Remove unwanted spaces
SELECT
  TRIM(cst_firstname) AS cst_firstname,
  TRIM(cst_lastname) AS cst_lastname
FROM `workspace`.`bronze`.`crm_cust_info`
;

-- Check results
SELECT
  cst_firstname,
  cst_lastname
FROM (
  SELECT
  TRIM(cst_firstname) AS cst_firstname,
  TRIM(cst_lastname) AS cst_lastname
  FROM `workspace`.`bronze`.`crm_cust_info`
)
WHERE
  cst_firstname != TRIM(cst_firstname) OR
  cst_lastname != TRIM(cst_lastname)
;

### 3 Standardization in `cst_marital_status` and `cst_gndr`

In [0]:
-- Check consistency of cst_marital_status
SELECT
  DISTINCT cst_marital_status  
FROM `workspace`.`bronze`.`crm_cust_info`;

-- Check consistency of cst_gndr
SELECT
  DISTINCT cst_gndr  
FROM `workspace`.`bronze`.`crm_cust_info`;

-- Change marital status and gender to a more user-friendly format
SELECT
  CASE WHEN upper(trim(cst_marital_status)) = 'S' THEN 'Single'
    WHEN upper(trim(cst_marital_status)) = 'M' THEN 'Married'
    ELSE 'n/a'
  END cst_marital_status,
  CASE WHEN upper(trim(cst_gndr)) = 'F' THEN 'Female'
    WHEN upper(trim(cst_gndr)) = 'M' THEN 'Male'
    ELSE 'n/a'
  END cst_gndr
FROM `workspace`.`bronze`.`crm_cust_info`
;

### Final query

In [0]:
-- Gathering transformations from steps 1 to 3
-- Adding the dwh_create_date with current_timestamp()
SELECT
  cst_id,
  cst_key,
  TRIM(cst_firstname) AS cst_firstname,
  TRIM(cst_lastname) AS cst_lastname,
  CASE WHEN upper(TRIM(cst_marital_status)) = 'S' THEN 'Single'
    WHEN upper(TRIM(cst_marital_status)) = 'M' THEN 'Married'
    ELSE 'n/a'
  END cst_marital_status,
  CASE WHEN upper(TRIM(cst_gndr)) = 'F' THEN 'Female'
    WHEN upper(TRIM(cst_gndr)) = 'M' THEN 'Male'
    ELSE 'n/a'
  END cst_gndr,
  cst_create_date,
  current_timestamp() AS dwh_create_date -- Adding the dwh_create_date
FROM (
  SELECT
  *,
  row_number() OVER(PARTITION BY cst_id ORDER BY cst_create_date DESC) AS flag_last
  FROM `workspace`.`bronze`.`crm_cust_info`
  )t
WHERE flag_last = 1
;

### Load into Silver

In [0]:
TRUNCATE TABLE silver.crm_cust_info; -- Garantees that the table is empty

INSERT INTO silver.crm_cust_info (
  cst_id, 
  cst_key, 
  cst_firstname, 
  cst_lastname, 
  cst_marital_status, 
  cst_gndr,
  cst_create_date,
  dwh_create_date
)
SELECT
  cst_id,
  cst_key,
  TRIM(cst_firstname) AS cst_firstname,
  TRIM(cst_lastname) AS cst_lastname,
  CASE 
    WHEN UPPER(TRIM(cst_marital_status)) = 'S' THEN 'Single'
    WHEN UPPER(TRIM(cst_marital_status)) = 'M' THEN 'Married'
    ELSE 'n/a'
  END AS cst_marital_status,
  CASE 
    WHEN UPPER(TRIM(cst_gndr)) = 'F' THEN 'Female'
    WHEN UPPER(TRIM(cst_gndr)) = 'M' THEN 'Male'
    ELSE 'n/a'
  END AS cst_gndr,
  cst_create_date,
  current_timestamp() AS dwh_create_date
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) AS flag_last
  FROM bronze.crm_cust_info
  WHERE cst_id IS NOT NULL
) t
WHERE flag_last = 1;


### Quality check

In [0]:
-- Primary key
SELECT
cst_id,
COUNT(*)
FROM silver.crm_cust_info
GROUP BY cst_id
HAVING COUNT(*) > 1 OR cst_id IS NULL
;

In [0]:
-- Unwanted spaces
SELECT
  cst_firstname,
  cst_lastname
FROM silver.crm_cust_info
WHERE
  cst_firstname != TRIM(cst_firstname) OR
  cst_lastname != TRIM(cst_lastname)
;


In [0]:
-- Check consistency of cst_marital_status
SELECT
  DISTINCT cst_marital_status  
FROM silver.crm_cust_info;

In [0]:
-- Check consistency of cst_gndr
SELECT
  DISTINCT cst_gndr  
FROM silver.crm_cust_info;

## crm_prd_info

### Explore the data

In [0]:
SELECT * FROM workspace.bronze.crm_prd_info;

-- We have to
-- 1 Check for duplicates and nulls in the primary key
-- 2 From prd_key, create cat_id (to join with erp_px_cat_g1v2)
-- 3 From prd_key, create new prd_key (to join with crm_sales_details)
-- 4 Check for unwanted spaces in prd_nm
-- 5 Check for negative numbers in prd_cost
-- 6 Standardize prd_line
-- 7 Check for consistency of prd_start_dt and prd_end_dt


### 1 Primary key check

In [0]:
-- 1 Checking for duplicates and nulls in the primary key
SELECT
prd_id,
COUNT(*)
FROM workspace.bronze.crm_prd_info
GROUP BY prd_id
HAVING COUNT(*) > 1 OR prd_id IS NULL
;

-- No duplicates or Nulls

### 2 Create `cat_id`

In [0]:
-- 2 Generate cat_id from prd_key

-- To join 'crm_prd_info' with 'erp_px_cat_g1v2' we will need a cat_id that matches
-- New column should look like 'AC_BR'

-- View erp_px_cat_g1v2
SELECT DISTINCT id FROM bronze.erp_px_cat_g1v2;

-- View crm_prd_info
SELECT
*
FROM workspace.bronze.crm_prd_info;

-- Generate cat_id from prd_key
SELECT
  prd_key,
  REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_') AS cat_id
FROM workspace.bronze.crm_prd_info;

### 3 Create compatible `prd_key`

In [0]:
-- 3 Generate compatible prd_key

-- To join 'crm_prd_info' with 'crm_sales_details' we will need a prd_key that matches
-- New column should look like 'BK-R93R-62'

-- View crm_sales_details
SELECT * FROM bronze.crm_sales_details;

-- View crm_prd_info
SELECT
*
FROM workspace.bronze.crm_prd_info;

-- Generate compatible prd_key
SELECT
  prd_key,
  SUBSTRING(prd_key, 7, LEN(prd_key)) AS prd_key
FROM workspace.bronze.crm_prd_info;

### 4 Unwanted spaces in `prd_nm`

In [0]:
-- Check for unwanted space
SELECT
  prd_nm
FROM bronze.crm_prd_info
WHERE prd_nm != TRIM(prd_nm)
;

-- No results

### 5 Consistency of `prd_cost`

In [0]:
-- Check for NULLs or negative numbers in prd_cost
SELECT
  prd_cost
FROM bronze.crm_prd_info
WHERE prd_cost < 0 OR prd_cost IS NULL
;

-- No negative numbers
-- Replace NULLs for 0
SELECT
  COALESCE(prd_cost, 0) AS prd_cost
FROM bronze.crm_prd_info
;

-- Check results
SELECT
  *
FROM (
  SELECT
  COALESCE(prd_cost, 0) AS prd_cost
FROM bronze.crm_prd_info
)t
WHERE prd_cost < 0 OR prd_cost IS NULL
;


### 6 Standardization in `prd_line`

In [0]:
-- Check data in column prd_line
SELECT DISTINCT prd_line
FROM bronze.crm_prd_info
;

-- Replace M, R, S, T and NULLs
SELECT
  prd_line,
  CASE 
    WHEN TRIM(prd_line) = 'M' THEN 'Mountain'
    WHEN TRIM(prd_line) = 'R' THEN 'Road'
    WHEN TRIM(prd_line) = 'S' THEN 'Other Sales'
    WHEN TRIM(prd_line) = 'T' THEN 'Touring'
    ELSE 'n/a'
  END AS prd_line
FROM bronze.crm_prd_info
;

### 7 Consistency of `prd_start_dt` and `prd_end_dt`

In [0]:
-- Check consistency of prd_start_dt and prd_end_dt
SELECT
  prd_key,
  prd_cost,
  prd_start_dt,
  prd_end_dt
FROM bronze.crm_prd_info
SORT BY prd_key
;

-- First, CAST as DATE as there are no time data
SELECT
  prd_key,
  prd_cost,
  CAST(prd_start_dt AS DATE),
  CAST(prd_end_dt AS DATE)
FROM bronze.crm_prd_info
SORT BY prd_key
;

-- Some times prd_end_dt < prd_start_dt
SELECT
  *
FROM bronze.crm_prd_info
WHERE prd_end_dt < prd_start_dt
SORT BY prd_key
;

-- Let's completely ignore the end date and build it from zero
-- For this we can use a window funcion

In [0]:
-- Let's focus on AC-HE-HL-U509
SELECT
  prd_key,
  prd_cost,
  CAST(prd_start_dt AS DATE),
  CAST(prd_end_dt AS DATE)
FROM bronze.crm_prd_info
WHERE prd_key = 'AC-HE-HL-U509'
;

-- Let's make a end_dt (for current row) with the start_dt from the following row
-- Step 1
-- substitute
-- prd_start_dt
-- with LEAD(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY prd_start_dt)
-- Step 2
-- subtract 1 day with DATE_SUB(, 1)
-- Step 3
-- Make sure we keep CAST( AS DATE)

-- Final query
SELECT
  prd_key,
  CAST(prd_start_dt AS DATE) AS prd_start_dt,
  CAST(
    DATE_SUB(LEAD(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY prd_start_dt), 1)
  AS DATE) AS prd_end_dt
FROM bronze.crm_prd_info
WHERE prd_key = 'AC-HE-HL-U509'
;

-- Check results
SELECT
*
FROM (
  SELECT
  prd_key,
  CAST(prd_start_dt AS DATE) AS prd_start_dt,
  CAST(
    DATE_SUB(LEAD(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY prd_start_dt), 1)
  AS DATE) AS prd_end_dt
  FROM bronze.crm_prd_info
)
WHERE prd_end_dt < prd_start_dt
;

### Final query

In [0]:
SELECT
  prd_id,
  REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_') AS cat_id,
  SUBSTRING(prd_key, 7, LEN(prd_key)) AS prd_key,
  prd_nm,
  COALESCE(prd_cost, 0) AS prd_cost,
    CASE 
    WHEN TRIM(prd_line) = 'M' THEN 'Mountain'
    WHEN TRIM(prd_line) = 'R' THEN 'Road'
    WHEN TRIM(prd_line) = 'S' THEN 'Other Sales'
    WHEN TRIM(prd_line) = 'T' THEN 'Touring'
    ELSE 'n/a'
  END AS prd_line,
  CAST(prd_start_dt AS DATE) AS prd_start_dt,
  CAST(
    DATE_SUB(LEAD(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY prd_start_dt), 1)
  AS DATE) AS prd_end_dt,
  current_timestamp() AS dwh_create_date -- ETL load timestamp
FROM bronze.crm_prd_info

### Load into Silver

In [0]:
-- Before loading, go to crm_prd_info Silver DDL notebook and:
-- 1 Add cat_id (STRING)
-- 2 Change datatype of   
--    prd_start_dt  (DATE)
--    prd_end_dt    (DATE)

In [0]:
TRUNCATE TABLE silver.crm_prd_info; -- Garantees that the table is empty

INSERT INTO silver.crm_prd_info(
  prd_id,
  cat_id,
  prd_key,
  prd_nm,
  prd_cost,
  prd_line,
  prd_start_dt,
  prd_end_dt,
  dwh_create_date
)
SELECT
  prd_id,
  REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_') AS cat_id,
  SUBSTRING(prd_key, 7, LEN(prd_key)) AS prd_key,
  prd_nm,
  COALESCE(prd_cost, 0) AS prd_cost,
    CASE 
    WHEN TRIM(prd_line) = 'M' THEN 'Mountain'
    WHEN TRIM(prd_line) = 'R' THEN 'Road'
    WHEN TRIM(prd_line) = 'S' THEN 'Other Sales'
    WHEN TRIM(prd_line) = 'T' THEN 'Touring'
    ELSE 'n/a'
  END AS prd_line,
  CAST(prd_start_dt AS DATE) AS prd_start_dt,
  CAST(
    DATE_SUB(LEAD(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY prd_start_dt), 1)
  AS DATE) AS prd_end_dt,
  current_timestamp() AS dwh_create_date -- ETL load timestamp
FROM bronze.crm_prd_info


### Quality check

In [0]:
-- Primary key
SELECT
prd_id,
COUNT(*)
FROM silver.crm_prd_info
GROUP BY prd_id
HAVING COUNT(*) > 1 OR prd_id IS NULL
;


In [0]:
-- New column cat_id
SELECT
  cat_id
FROM silver.crm_prd_info
GROUP BY cat_id
;

In [0]:
-- Column prd_key
SELECT
  prd_key
FROM silver.crm_prd_info
GROUP BY prd_key
ORDER BY prd_key
;

In [0]:
-- Check for NULLs or negative numbers in prd_cost
SELECT
  prd_cost
FROM silver.crm_prd_info
WHERE prd_cost < 0 OR prd_cost IS NULL
;

In [0]:
-- Check data in column prd_line
SELECT DISTINCT prd_line
FROM silver.crm_prd_info
;

In [0]:
-- Check consistency of prd_start_dt and prd_end_dt
SELECT
  *
FROM silver.crm_prd_info
WHERE prd_end_dt < prd_start_dt
SORT BY prd_key
;

## crm_sales_details

### Explore the data

In [0]:
SELECT * FROM workspace.bronze.crm_sales_details;

-- We have to
-- 1 Check for unwanted spaces in sls_ord_num
-- 2 Check integrity of sls_prd_key and sls_cust_id
-- 3 Check integrity of 3 date columns
-- 4 Check integrity of 3 sales columns


### 1 Unwanted spaces in sls_ord_num

In [0]:
-- Unwanted spaces in sls_ord_num
SELECT
  *
FROM workspace.bronze.crm_sales_details
WHERE sls_ord_num != TRIM(sls_ord_num)
;

-- No issues

### 2 Check keys integrity

In [0]:
-- Integrity of sls_prd_key
SELECT
  *
FROM workspace.bronze.crm_sales_details
WHERE sls_prd_key NOT IN (SELECT prd_key FROM silver.crm_prd_info)
;
-- No issues

In [0]:
-- Integrity of sls_cust_id
SELECT
  *
FROM workspace.bronze.crm_sales_details
WHERE sls_cust_id NOT IN (SELECT cst_id FROM silver.crm_cust_info)
;

-- No issues

### 3 Check date columns

In [0]:
-- Check column sls_order_dt

-- Dates are stored as INT
-- Also, we have to check for boundary conditions
SELECT
  sls_order_dt
FROM workspace.bronze.crm_sales_details
WHERE
  sls_order_dt <= 0 OR
  LEN(sls_order_dt) != 8 OR
  sls_order_dt > 20300101 OR
  sls_order_dt < 19000101
;

-- 1 Change exceptions to Null
-- 2 Use to_date and CAST to convert to DATE
SELECT
CASE WHEN sls_order_dt = 0 OR LEN(sls_order_dt) != 8 THEN NULL
  ELSE to_date(CAST(sls_order_dt AS STRING), 'yyyyMMdd')
END AS sls_order_dt
FROM workspace.bronze.crm_sales_details
;

In [0]:
-- Check column sls_ship_dt
SELECT
  sls_ship_dt
FROM workspace.bronze.crm_sales_details
WHERE
  sls_ship_dt <= 0 OR
  LEN(sls_ship_dt) != 8 OR
  sls_ship_dt > 20300101 OR
  sls_ship_dt < 19000101
;

-- No issues

In [0]:
-- Check column sls_ship_dt
SELECT
  sls_due_dt
FROM workspace.bronze.crm_sales_details
WHERE
  sls_due_dt <= 0 OR
  LEN(sls_due_dt) != 8 OR
  sls_due_dt > 20300101 OR
  sls_due_dt < 19000101
;

-- No issues

In [0]:
-- All dates transformations

SELECT
CASE WHEN sls_order_dt = 0 OR LEN(sls_order_dt) != 8 THEN NULL
  ELSE to_date(CAST(sls_order_dt AS STRING), 'yyyyMMdd')
END AS sls_order_dt,
to_date(CAST(sls_ship_dt AS STRING), 'yyyyMMdd') AS sls_ship_dt,
to_date(CAST(sls_due_dt AS STRING), 'yyyyMMdd') AS sls_due_dt
FROM workspace.bronze.crm_sales_details
;

### 4 Check sales data integrity

In [0]:
-- See columns
SELECT * FROM workspace.bronze.crm_sales_details;

-- Check integrity
SELECT
  sls_sales,
  sls_quantity,
  sls_price
FROM workspace.bronze.crm_sales_details
WHERE
  sls_sales != sls_quantity * sls_price OR
  sls_sales IS NULL OR sls_quantity IS NULL OR sls_price IS NULL OR
  sls_sales <= 0 OR sls_quantity <= 0 OR sls_price <= 0
ORDER BY sls_sales, sls_quantity, sls_price
;

-- Column sls_quantity is fine
-- Columns sls_sales and sls_price contains NULL, zeros and negative values

In [0]:
-- Fix data on sls_sales and sls_price
SELECT
  CASE WHEN sls_sales IS NULL OR sls_sales <= 0 OR sls_sales != sls_quantity * ABS(sls_price)
        THEN sls_quantity * ABS(sls_price)
        ELSE sls_sales
  END AS sls_sales,
  sls_quantity,
  CASE WHEN sls_price IS NULL OR sls_price <= 0
        THEN sls_sales / NULLIF(sls_quantity, 0)
        ELSE sls_price
  END AS sls_price
FROM workspace.bronze.crm_sales_details
;

-- Check results
SELECT
  sls_sales,
  sls_quantity,
  sls_price
FROM 
  (
    SELECT
    CASE WHEN sls_sales IS NULL OR sls_sales <= 0 OR sls_sales != sls_quantity * ABS(sls_price)
          THEN sls_quantity * ABS(sls_price)
          ELSE sls_sales
    END AS sls_sales,
    sls_quantity,
    CASE WHEN sls_price IS NULL OR sls_price <= 0
          THEN sls_sales / NULLIF(sls_quantity, 0)
          ELSE sls_price
    END AS sls_price
  FROM workspace.bronze.crm_sales_details
  )
WHERE
  sls_sales != sls_quantity * sls_price OR
  sls_sales IS NULL OR sls_quantity IS NULL OR sls_price IS NULL OR
  sls_sales <= 0 OR sls_quantity <= 0 OR sls_price <= 0
ORDER BY sls_sales, sls_quantity, sls_price
;



### Final query

In [0]:
SELECT
  sls_ord_num, -- ok
  sls_prd_key, -- ok
  sls_cust_id, -- ok
  CASE WHEN sls_order_dt = 0 OR LEN(sls_order_dt) != 8 THEN NULL
    ELSE to_date(CAST(sls_order_dt AS STRING), 'yyyyMMdd')
  END AS sls_order_dt,
  to_date(CAST(sls_ship_dt AS STRING), 'yyyyMMdd') AS sls_ship_dt,
  to_date(CAST(sls_due_dt AS STRING), 'yyyyMMdd') AS sls_due_dt,
  CASE WHEN sls_sales IS NULL OR sls_sales <= 0 OR sls_sales != sls_quantity * ABS(sls_price)
        THEN sls_quantity * ABS(sls_price)
        ELSE sls_sales
  END AS sls_sales,
  sls_quantity, -- ok
  CASE WHEN sls_price IS NULL OR sls_price <= 0
        THEN sls_sales / NULLIF(sls_quantity, 0)
        ELSE sls_price
  END AS sls_price
FROM workspace.bronze.crm_sales_details


### Load into Silver

In [0]:
-- Before loading, go to crm_sls_details Silver DDL notebook and:
-- Change datatype of   
--    sls_order_dt  (DATE)
--    sls_ship_dt   (DATE)
--    sls_due_dt    (DATE)

-- Run DDL

In [0]:
TRUNCATE TABLE silver.crm_sales_details; -- Garantees that the table is empty

INSERT INTO silver.crm_sales_details(
  sls_ord_num,
  sls_prd_key,
  sls_cust_id,
  sls_order_dt,
  sls_ship_dt,
  sls_due_dt,
  sls_sales,
  sls_quantity,
  sls_price,
  dwh_create_date
)
SELECT
  sls_ord_num,
  sls_prd_key,
  sls_cust_id,
  CASE WHEN sls_order_dt = 0 OR LEN(sls_order_dt) != 8 THEN NULL
    ELSE to_date(CAST(sls_order_dt AS STRING), 'yyyyMMdd')
  END AS sls_order_dt,
  to_date(CAST(sls_ship_dt AS STRING), 'yyyyMMdd') AS sls_ship_dt,
  to_date(CAST(sls_due_dt AS STRING), 'yyyyMMdd') AS sls_due_dt,
  CASE WHEN sls_sales IS NULL OR sls_sales <= 0 OR sls_sales != sls_quantity * ABS(sls_price)
        THEN sls_quantity * ABS(sls_price)
        ELSE sls_sales
  END AS sls_sales,
  sls_quantity,
  CASE WHEN sls_price IS NULL OR sls_price <= 0
        THEN sls_sales / NULLIF(sls_quantity, 0)
        ELSE sls_price
  END AS sls_price,
  current_timestamp() AS dwh_create_date -- ETL load timestamp
FROM workspace.bronze.crm_sales_details


### Quality check

In [0]:
-- Check column sls_order_dt
SELECT
  sls_order_dt
FROM silver.crm_sales_details
WHERE
  sls_order_dt < DATE('1900-01-01') OR
  sls_order_dt > DATE('2030-01-01') OR
  sls_order_dt IS NULL;

-- We are not going to fix this column

In [0]:
-- Check sales data
SELECT
  sls_sales,
  sls_quantity,
  sls_price
FROM silver.crm_sales_details
WHERE
  sls_sales != sls_quantity * sls_price OR
  sls_sales IS NULL OR sls_quantity IS NULL OR sls_price IS NULL OR
  sls_sales <= 0 OR sls_quantity <= 0 OR sls_price <= 0
ORDER BY sls_sales, sls_quantity, sls_price
;


## erp_cust_az12

### Explore data

In [0]:
SELECT * FROM workspace.bronze.erp_cust_az12 LIMIT 10;

-- We have to:
-- 1 Transform cid to match cst_key from crm_cust_info
-- 2 Check integrity of bdate
-- 3 Check consistency of gen

### 1 Transform `cid`

In [0]:
-- Check crm_cust_info table
SELECT * FROM workspace.silver.crm_cust_info LIMIT 5;

-- Check length of cst_key
SELECT
  LEN(cst_key),
  COUNT(LEN(cst_key))
FROM workspace.silver.crm_cust_info
GROUP BY LEN(cst_key)
;

In [0]:
-- 1 Transform cid to match cst_id from crm_cust_info
SELECT
  LEN(cid),
  COUNT(LEN(cid))
FROM workspace.bronze.erp_cust_az12
GROUP BY LEN(cid)
;

-- See examples of cid with LEN = 13
SELECT
  cid
FROM workspace.bronze.erp_cust_az12
WHERE LEN(cid) = 13
;

-- Some cid have 'NAS' on the start. Let's remove it
SELECT
  CASE WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid, 4, LEN(cid))
    ELSE cid
  END AS cid
FROM workspace.bronze.erp_cust_az12
;

-- Check results adding WHERE clause to see if all new cid are contained in crm_cust_info
SELECT
  CASE WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid, 4, LEN(cid))
    ELSE cid
  END AS cid
FROM workspace.bronze.erp_cust_az12
WHERE
  CASE WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid, 4, LEN(cid))
    ELSE cid
  END NOT IN (SELECT DISTINCT cst_key FROM workspace.silver.crm_cust_info)
;

### 2 Integrity of bdate

In [0]:
-- Check bdate
SELECT
  bdate
FROM workspace.bronze.erp_cust_az12
WHERE bdate < DATE('1900-01-01') OR bdate > getdate()
;

-- Remove dates in the future
SELECT
  CASE WHEN bdate > getdate() THEN NULL
    ELSE bdate
  END AS bdate
FROM workspace.bronze.erp_cust_az12
;

-- Check results
SELECT
  bdate
FROM
  (
   SELECT
  CASE WHEN bdate > getdate() THEN NULL
    ELSE bdate
  END AS bdate
  FROM workspace.bronze.erp_cust_az12
  )
WHERE bdate < DATE('1900-01-01') OR bdate > getdate()
;

### 3 Check consistency of `gen`

In [0]:
-- Check gen column
SELECT DISTINCT gen
FROM workspace.bronze.erp_cust_az12
;

-- Transform
SELECT
  CASE WHEN UPPER(TRIM(gen)) IN ('M', 'MALE') THEN 'Male'
    WHEN UPPER(TRIM(gen)) IN ('F', 'FEMALE') THEN 'Female'
    ELSE 'n/a'
  END AS gen
FROM workspace.bronze.erp_cust_az12
;

-- Check results
SELECT DISTINCT gen
FROM 
  (
    SELECT
  CASE WHEN UPPER(TRIM(gen)) IN ('M', 'MALE') THEN 'Male'
    WHEN UPPER(TRIM(gen)) IN ('F', 'FEMALE') THEN 'Female'
    ELSE 'n/a'
  END AS gen
  FROM workspace.bronze.erp_cust_az12
  )
;

### Final query

In [0]:
-- Final query
SELECT
  CASE WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid, 4, LEN(cid))
    ELSE cid
  END AS cid,
    CASE WHEN bdate > getdate() THEN NULL
    ELSE bdate
  END AS bdate,
  CASE WHEN UPPER(TRIM(gen)) IN ('M', 'MALE') THEN 'Male'
    WHEN UPPER(TRIM(gen)) IN ('F', 'FEMALE') THEN 'Female'
    ELSE 'n/a'
  END AS gen
FROM workspace.bronze.erp_cust_az12
;

### Load into Silver

In [0]:
TRUNCATE TABLE silver.erp_cust_az12; -- Garantees that the table is empty

INSERT INTO silver.erp_cust_az12(
  cid,
  bdate,
  gen,
  dwh_create_date
)
SELECT
  CASE WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid, 4, LEN(cid))
    ELSE cid
  END AS cid,
    CASE WHEN bdate > getdate() THEN NULL
    ELSE bdate
  END AS bdate,
  CASE WHEN UPPER(TRIM(gen)) IN ('M', 'MALE') THEN 'Male'
    WHEN UPPER(TRIM(gen)) IN ('F', 'FEMALE') THEN 'Female'
    ELSE 'n/a'
  END AS gen,
  current_timestamp() AS dwh_create_date -- ETL load timestamp
FROM workspace.bronze.erp_cust_az12
;

### Quality check

In [0]:
-- Check cid
SELECT
  LEN(cid),
  COUNT(LEN(cid))
FROM silver.erp_cust_az12
GROUP BY LEN(cid)
;

In [0]:
-- Check bdate
SELECT
  bdate
FROM silver.erp_cust_az12
WHERE bdate < DATE('1900-01-01') OR bdate > getdate()
;

In [0]:
-- Check gen
SELECT DISTINCT gen
FROM silver.erp_cust_az12
;

## erp_loc_a101

### Explore data

In [0]:
SELECT * FROM workspace.bronze.erp_loc_a101 LIMIT 10;

-- We have to:
-- 1 Transform cid to match cst_key from crm_cust_info
-- 2 Check consistency of cntry

### 1 Transform `cid`

In [0]:
-- Check crm_cust_info table
SELECT * FROM workspace.silver.crm_cust_info LIMIT 5;

-- Replace '-' with '' in cid
SELECT
  REPLACE(cid, '-', '') AS cid
FROM workspace.bronze.erp_loc_a101
;

-- Check if mathces
SELECT
  REPLACE(cid, '-', '') AS cid
FROM workspace.bronze.erp_loc_a101
WHERE
  REPLACE(cid, '-', '') NOT IN (
  SELECT cst_key FROM workspace.silver.crm_cust_info
)
;

### 2 Consistency of `cntry`

In [0]:
-- Check all distinct values
SELECT DISTINCT cntry
FROM workspace.bronze.erp_loc_a101
ORDER BY cntry
;

-- Standardize data
SELECT
CASE WHEN TRIM(cntry) = 'DE' THEN 'Germany'
      WHEN TRIM(cntry) IN ('US', 'USA') THEN 'United States'
      WHEN TRIM(cntry) = '' OR cntry IS NULL THEN 'n/a'
      ELSE TRIM(cntry)
END AS cntry
FROM workspace.bronze.erp_loc_a101
;

-- Check results
SELECT DISTINCT cntry
FROM
  (
    SELECT
    CASE WHEN TRIM(cntry) = 'DE' THEN 'Germany'
          WHEN TRIM(cntry) IN ('US', 'USA') THEN 'United States'
          WHEN TRIM(cntry) = '' OR cntry IS NULL THEN 'n/a'
          ELSE TRIM(cntry)
    END AS cntry
    FROM workspace.bronze.erp_loc_a101
  )
ORDER BY cntry
;

### Final query

In [0]:
-- Final query
SELECT
  REPLACE(cid, '-', '') AS cid,
  CASE WHEN TRIM(cntry) = 'DE' THEN 'Germany'
      WHEN TRIM(cntry) IN ('US', 'USA') THEN 'United States'
      WHEN TRIM(cntry) = '' OR cntry IS NULL THEN 'n/a'
      ELSE TRIM(cntry)
  END AS cntry
FROM workspace.bronze.erp_loc_a101
;

### Load into Silver

In [0]:
TRUNCATE TABLE workspace.silver.erp_loc_a101; -- Garantees that the table is empty

INSERT INTO workspace.silver.erp_loc_a101(
  cid,
  cntry,
  dwh_create_date
)
SELECT
  REPLACE(cid, '-', '') AS cid,
  CASE WHEN TRIM(cntry) = 'DE' THEN 'Germany'
      WHEN TRIM(cntry) IN ('US', 'USA') THEN 'United States'
      WHEN TRIM(cntry) = '' OR cntry IS NULL THEN 'n/a'
      ELSE TRIM(cntry)
  END AS cntry,
  current_timestamp() AS dwh_create_date -- ETL load timestamp
FROM workspace.bronze.erp_loc_a101
;

### Quality check

In [0]:
-- Check if cid mathces
SELECT
  cid
FROM workspace.silver.erp_loc_a101
WHERE
  cid NOT IN (
  SELECT cst_key FROM workspace.silver.crm_cust_info
)
;

In [0]:
-- Check cntry values
SELECT DISTINCT cntry
FROM workspace.silver.erp_loc_a101
ORDER BY cntry
;

## erp_px_cat_g1v2

### Explore data

In [0]:
SELECT * FROM workspace.bronze.erp_px_cat_g1v2 LIMIT 10;

-- Column id is already matching with column cat_id from crm_prd_info

-- We have to:
-- 1 Check for unwanted spaces
-- 2 Check data consistancy

### 1 Unwanted spaces

In [0]:
-- Check columns for unwanted spaces
SELECT
  *
FROM workspace.bronze.erp_px_cat_g1v2
WHERE
  TRIM(cat) != cat OR
  TRIM(subcat) != subcat OR
  TRIM(maintenance) != maintenance
;

### 2 Check consistency of columns

In [0]:
-- Check consisteyncy of column cat
SELECT DISTINCT
  cat
FROM workspace.bronze.erp_px_cat_g1v2
;

In [0]:
-- Check consisteyncy of column subcat
SELECT DISTINCT
  subcat
FROM workspace.bronze.erp_px_cat_g1v2
;

In [0]:
-- Check consisteyncy of column maintenance
SELECT DISTINCT
  maintenance
FROM workspace.bronze.erp_px_cat_g1v2
;

### Final query

In [0]:
-- Check consisteyncy of column maintenance
SELECT
  cat,
  subcat,
  maintenance
FROM workspace.bronze.erp_px_cat_g1v2
;

### Load into Silver

In [0]:
TRUNCATE TABLE workspace.silver.erp_px_cat_g1v2; -- Garantees that the table is empty

INSERT INTO workspace.silver.erp_px_cat_g1v2(
  cat,
  subcat,
  maintenance,
  dwh_create_date
)
SELECT
  cat,
  subcat,
  maintenance,
  current_timestamp() AS dwh_create_date -- ETL load timestamp
FROM workspace.bronze.erp_px_cat_g1v2
;